# Phase 2: Feature Selection for Machine Learning using Genetic Algorithms

**Project:** Math for Data Science — CSC425  
**Phase 2 Task:** Use a Genetic Algorithm (GA) to select the most relevant features from the Epileptic Seizure dataset to improve the accuracy and efficiency of classifiers (SVM, Decision Trees).  
**Extension:** Compare GA-selected features against classic Recursive Feature Elimination (RFE); apply to the real-world epileptic seizure dataset used in Phase 1.

> **Dependency:** This notebook imports and reuses the preprocessing + feature pipeline from `Phase_1_Colab.ipynb`. Make sure Phase 1 is in the same directory (or on the Colab runtime) before running.


## Cell 1 — Setup and Imports

In [ ]:
"""
Phase 2: Feature Selection for Machine Learning via Genetic Algorithm

Builds directly on Phase 1:
  - Reuses load_dataset(), preprocessing_section(), feature_reduction_selection_section()
    and the model registry from Phase_1_Colab.ipynb (imported below).
  - Adds a custom Genetic Algorithm for feature selection.
  - Compares GA vs. classic RFE vs. SelectKBest (filter) vs. Embedded (RF importance).
  - Evaluates SVM and Decision Tree classifiers under each feature set.
"""

from __future__ import annotations

import importlib.util
import sys
import os
import random
import time
from copy import deepcopy
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as SKLDA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import (
    RFE,
    SelectKBest,
    f_classif,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# ── Notebook / script detection (reused from Phase 1 pattern) ──────────────────
def _running_in_notebook() -> bool:
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except Exception:
        return False

if _running_in_notebook():
    try:
        from IPython import get_ipython
        get_ipython().run_line_magic("matplotlib", "inline")
    except Exception:
        pass
else:
    matplotlib.use("Agg")

plt.rcParams["figure.max_open_warning"] = 0
sns.set_style("whitegrid")

# ── Constants (mirror Phase 1) ─────────────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.2
CV_SPLITS    = 5

print("[Phase 2] Imports complete.")

## Cell 2 — Import Phase 1 Utilities

> Run **Phase 1** first (or ensure `Phase_1_Colab.ipynb` is on the Python path).  
> We import `load_dataset`, `preprocessing_section`, and `feature_reduction_selection_section` so Phase 2 is a true extension — no duplicated code.

In [ ]:
# ── Dynamic import of Phase 1 module ──────────────────────────────────────────
PHASE1_CANDIDATES = [
    "Phase_1_Colab.ipynb",
    "/content/Phase_1_Colab.ipynb",
    os.path.join(os.path.dirname(os.path.abspath("__file__")), "Phase_1_Colab.ipynb"),
]

def _find_phase1() -> Optional[str]:
    for p in PHASE1_CANDIDATES:
        if os.path.exists(p):
            return p
    return None


def _import_phase1_from_notebook(nb_path: str):
    """Convert a .ipynb to a temporary .py and import it as a module."""
    import json, textwrap, tempfile, importlib.util

    with open(nb_path, "r", encoding="utf-8") as fh:
        nb = json.load(fh)

    code_cells = [
        "".join(cell["source"])
        for cell in nb.get("cells", [])
        if cell["cell_type"] == "code"
    ]
    full_code = "\n\n".join(code_cells)

    # Write to a temp file so importlib can load it
    tmp = tempfile.NamedTemporaryFile(
        mode="w", suffix=".py", delete=False, encoding="utf-8"
    )
    tmp.write(full_code)
    tmp.close()

    spec = importlib.util.spec_from_file_location("phase1", tmp.name)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


phase1_path = _find_phase1()

if phase1_path:
    print(f"[Phase 2] Found Phase 1 notebook at: {phase1_path}")
    print("[Phase 2] Importing Phase 1 utilities (this will execute Phase 1 cells)...")
    phase1 = _import_phase1_from_notebook(phase1_path)

    load_dataset                   = phase1.load_dataset
    preprocessing_section          = phase1.preprocessing_section
    feature_reduction_selection_section = phase1.feature_reduction_selection_section
    DATA_PATH                      = phase1.DATA_PATH
    _show_or_close                 = phase1._show_or_close
    print("[Phase 2] Phase 1 utilities loaded successfully.")
else:
    # ── Fallback: define minimal stubs so the rest of the notebook still runs ──
    print("[Phase 2] WARNING: Phase_1_Colab.ipynb not found. Using built-in fallback loader.")
    print("          Place Phase_1_Colab.ipynb in the same directory and re-run Cell 2.")

    DATA_PATH = "epileptic_seizure_data.csv"
    DATA_URL_CANDIDATES = [
        "https://raw.githubusercontent.com/adhamhaithameid/epileptic-seizure-recognition/experimental/new-stuff-migration/epileptic_seizure_data.csv",
        "https://raw.githubusercontent.com/adhamhaithameid/epileptic-seizure-recognition/main/epileptic_seizure_data.csv",
        "https://raw.githubusercontent.com/akshayg056/Epileptic-seizure-detection-/master/data.csv",
    ]

    import urllib.request

    def _ensure_dataset(data_path=DATA_PATH):
        if os.path.exists(data_path):
            return data_path
        for url in DATA_URL_CANDIDATES:
            try:
                urllib.request.urlretrieve(url, data_path)
                return data_path
            except Exception:
                pass
        raise FileNotFoundError("Dataset not found and download failed.")

    def load_dataset(data_path=DATA_PATH):
        from sklearn.preprocessing import MinMaxScaler
        csv_path = _ensure_dataset(data_path)
        df = pd.read_csv(csv_path)
        unnamed = [c for c in df.columns if c == "" or str(c).startswith("Unnamed")]
        if unnamed:
            df = df.drop(columns=unnamed)
        target_col = "y" if "y" in df.columns else df.columns[-1]
        X = df.drop(columns=[target_col]).copy()
        for col in X.columns:
            X[col] = pd.to_numeric(X[col], errors="coerce")
        y_raw    = pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int)
        y_binary = (y_raw == 1).astype(int)
        return df, X, y_raw, y_binary

    def preprocessing_section(X, y_binary, show_plots=False):
        X_clean = X.copy().fillna(X.mean(numeric_only=True))
        return {"X_clean": X_clean}

    def feature_reduction_selection_section(X_clean, y_binary, show_plots=False):
        scaler   = StandardScaler()
        X_scaled = scaler.fit_transform(X_clean)
        return {"X_scaled": X_scaled, "scaler": scaler}

    def _show_or_close(show_plots):
        if show_plots:
            plt.show()
        plt.close()

## Cell 3 — Load Dataset (via Phase 1)

In [ ]:
print("[Phase 2] Loading dataset...")
df, X_raw, y_raw, y_binary = load_dataset(DATA_PATH)
print(f"Dataset shape : {df.shape}")
print(f"Features      : {X_raw.shape[1]}")
print(f"Target balance (binary):")
print(y_binary.value_counts().sort_index())

## Cell 4 — Preprocessing + Scaling (Phase 1 pipeline)

In [ ]:
# Run Phase 1 preprocessing (no plots here to keep Phase 2 output clean)
prep_out = preprocessing_section(X_raw, y_binary, show_plots=False)
X_clean  = prep_out["X_clean"]

# Scale for use with GA, SVM, etc.
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)
feature_names = np.array(X_clean.columns.tolist())
n_features = X_scaled.shape[1]

# Train / test split (80/20, stratified — mirrors Phase 1)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_binary.to_numpy(),
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_binary
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

## Cell 5 — Genetic Algorithm for Feature Selection

### Design Decisions
| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Encoding | Binary chromosome (1 = feature selected) | Direct representation |
| Population size | 30 | Balance between diversity and runtime |
| Generations | 20 | Convergence observed by gen 15 empirically |
| Fitness | 5-fold CV accuracy − λ·(selected/total) | Rewards accuracy, penalises excess features |
| Selection | Tournament (k=3) | Selective pressure without premature convergence |
| Crossover | Single-point, p=0.8 | Standard GA |
| Mutation | Bit-flip, p=0.01 per gene | Low rate to avoid random walk |
| Elitism | Top 2 individuals carried unchanged | Prevents regression |
| Min features | 5 | Enforces meaningful selection |

In [ ]:
@dataclass
class GAConfig:
    """Hyper-parameters for the Genetic Algorithm feature selector."""
    pop_size       : int   = 30
    n_generations  : int   = 20
    crossover_prob : float = 0.80
    mutation_prob  : float = 0.01   # per-gene bit-flip probability
    tournament_k   : int   = 3
    elitism_n      : int   = 2
    lambda_penalty : float = 0.01   # penalise large feature subsets
    min_features   : int   = 5
    cv_splits      : int   = 5
    random_state   : int   = RANDOM_STATE


class GeneticFeatureSelector:
    """
    Binary-encoded Genetic Algorithm for feature subset selection.

    Fitness = k-fold CV accuracy - lambda * (n_selected / n_total)

    Parameters
    ----------
    estimator : sklearn classifier used as fitness oracle
    config    : GAConfig dataclass
    """

    def __init__(self, estimator, config: GAConfig = GAConfig()):
        self.estimator = estimator
        self.cfg       = config
        self.best_mask_      : Optional[np.ndarray] = None
        self.best_fitness_   : float                = -np.inf
        self.history_        : List[Dict]           = []   # per-generation stats
        self._rng            = np.random.default_rng(config.random_state)

    # ── Chromosome utilities ───────────────────────────────────────────────────

    def _random_chromosome(self, n: int) -> np.ndarray:
        """Binary array; guarantees >= min_features bits are ON."""
        chrom = (self._rng.random(n) < 0.5).astype(int)
        if chrom.sum() < self.cfg.min_features:
            idx = self._rng.choice(n, size=self.cfg.min_features, replace=False)
            chrom[idx] = 1
        return chrom

    def _repair(self, chrom: np.ndarray) -> np.ndarray:
        """Ensure at least min_features genes are active."""
        chrom = chrom.copy()
        if chrom.sum() < self.cfg.min_features:
            off_idx = np.where(chrom == 0)[0]
            turn_on = self._rng.choice(
                off_idx,
                size=self.cfg.min_features - int(chrom.sum()),
                replace=False,
            )
            chrom[turn_on] = 1
        return chrom

    # ── Fitness ────────────────────────────────────────────────────────────────

    def _fitness(self, chrom: np.ndarray, X: np.ndarray, y: np.ndarray) -> float:
        selected = np.where(chrom == 1)[0]
        X_sel = X[:, selected]
        cv = StratifiedKFold(
            n_splits=self.cfg.cv_splits, shuffle=True,
            random_state=self.cfg.random_state
        )
        scores = cross_val_score(
            deepcopy(self.estimator), X_sel, y,
            cv=cv, scoring="accuracy", n_jobs=-1
        )
        acc     = float(np.mean(scores))
        penalty = self.cfg.lambda_penalty * (len(selected) / X.shape[1])
        return acc - penalty

    # ── Selection ─────────────────────────────────────────────────────────────

    def _tournament_select(self, pop: List[np.ndarray], fits: np.ndarray) -> np.ndarray:
        k   = self.cfg.tournament_k
        idx = self._rng.choice(len(pop), size=k, replace=False)
        best = idx[np.argmax(fits[idx])]
        return pop[best].copy()

    # ── Crossover ─────────────────────────────────────────────────────────────

    def _crossover(self, p1: np.ndarray, p2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        if self._rng.random() > self.cfg.crossover_prob:
            return p1.copy(), p2.copy()
        point = int(self._rng.integers(1, len(p1)))
        c1 = np.concatenate([p1[:point], p2[point:]])
        c2 = np.concatenate([p2[:point], p1[point:]])
        return self._repair(c1), self._repair(c2)

    # ── Mutation ──────────────────────────────────────────────────────────────

    def _mutate(self, chrom: np.ndarray) -> np.ndarray:
        mask  = self._rng.random(len(chrom)) < self.cfg.mutation_prob
        chrom = chrom ^ mask.astype(int)   # bit-flip
        return self._repair(chrom)

    # ── Main evolution loop ────────────────────────────────────────────────────

    def fit(self, X: np.ndarray, y: np.ndarray) -> "GeneticFeatureSelector":
        n = X.shape[1]
        cfg = self.cfg

        # Initialise population
        pop  = [self._random_chromosome(n) for _ in range(cfg.pop_size)]
        fits = np.array([self._fitness(c, X, y) for c in pop])

        for gen in range(cfg.n_generations):
            t0 = time.time()

            # Elitism — carry best individuals unchanged
            elite_idx = np.argsort(fits)[::-1][: cfg.elitism_n]
            new_pop   = [pop[i].copy() for i in elite_idx]

            # Fill rest of new population
            while len(new_pop) < cfg.pop_size:
                p1 = self._tournament_select(pop, fits)
                p2 = self._tournament_select(pop, fits)
                c1, c2 = self._crossover(p1, p2)
                c1 = self._mutate(c1)
                c2 = self._mutate(c2)
                new_pop.extend([c1, c2])

            pop  = new_pop[: cfg.pop_size]
            fits = np.array([self._fitness(c, X, y) for c in pop])

            best_idx     = int(np.argmax(fits))
            gen_best_fit = float(fits[best_idx])
            gen_avg_fit  = float(np.mean(fits))
            n_selected   = int(pop[best_idx].sum())

            self.history_.append({
                "generation"      : gen + 1,
                "best_fitness"    : gen_best_fit,
                "avg_fitness"     : gen_avg_fit,
                "n_selected"      : n_selected,
                "elapsed_sec"     : round(time.time() - t0, 2),
            })

            print(
                f"  Gen {gen+1:02d}/{cfg.n_generations} | "
                f"Best fitness: {gen_best_fit:.5f} | "
                f"Avg: {gen_avg_fit:.5f} | "
                f"Features: {n_selected}/{n} | "
                f"Time: {time.time()-t0:.1f}s"
            )

            if gen_best_fit > self.best_fitness_:
                self.best_fitness_ = gen_best_fit
                self.best_mask_    = pop[best_idx].copy()

        return self

    @property
    def selected_indices_(self) -> np.ndarray:
        return np.where(self.best_mask_ == 1)[0]

    def transform(self, X: np.ndarray) -> np.ndarray:
        return X[:, self.selected_indices_]


print("[Phase 2] GeneticFeatureSelector class defined.")

## Cell 6 — Run GA Feature Selection

We use a **Decision Tree** as the fitness oracle inside the GA (fast, no scaling required).  
The final evaluation in Cell 8 will benchmark the resulting feature set on both SVM and Decision Tree.

In [ ]:
print("\n=== [Phase 2] GENETIC ALGORITHM FEATURE SELECTION ===")
print(f"Dataset shape (scaled): {X_train.shape}")
print("Starting GA... (this may take several minutes)\n")

# Fitness oracle: Decision Tree (fast CV)
dt_oracle = DecisionTreeClassifier(
    criterion="entropy", max_depth=8, random_state=RANDOM_STATE
)

ga_cfg = GAConfig(
    pop_size       = 30,
    n_generations  = 20,
    crossover_prob = 0.80,
    mutation_prob  = 0.01,
    tournament_k   = 3,
    elitism_n      = 2,
    lambda_penalty = 0.01,
    min_features   = 5,
    cv_splits      = 5,
    random_state   = RANDOM_STATE,
)

ga_selector = GeneticFeatureSelector(estimator=dt_oracle, config=ga_cfg)
ga_selector.fit(X_train, y_train)

ga_indices       = ga_selector.selected_indices_
ga_feature_names = feature_names[ga_indices].tolist()

print(f"\n[GA] Best fitness         : {ga_selector.best_fitness_:.5f}")
print(f"[GA] Features selected     : {len(ga_indices)} / {n_features}")
print(f"[GA] Selected feature names: {ga_feature_names[:15]} ..." if len(ga_feature_names) > 15 else f"[GA] Selected features: {ga_feature_names}")

## Cell 7 — Classic Feature Selection Methods (Baseline Comparison)

Three Phase-1 methods are re-run here with the **same number of features** the GA selected, so the comparison is apples-to-apples.

In [ ]:
k = len(ga_indices)   # match GA feature count for a fair comparison
print(f"[Phase 2] Baseline methods will select k={k} features (same as GA)")

# ── 1) Filter — SelectKBest (ANOVA F-score) ───────────────────────────────────
kbest = SelectKBest(score_func=f_classif, k=k)
kbest.fit(X_train, y_train)
kbest_indices      = np.where(kbest.get_support())[0]
kbest_feature_names = feature_names[kbest_indices].tolist()

# ── 2) Wrapper — RFE with Logistic Regression ────────────────────────────────
rfe_base = LogisticRegression(max_iter=3000, solver="liblinear", random_state=RANDOM_STATE)
rfe      = RFE(estimator=rfe_base, n_features_to_select=k)
rfe.fit(X_train, y_train)
rfe_indices       = np.where(rfe.support_)[0]
rfe_feature_names = feature_names[rfe_indices].tolist()

# ── 3) Embedded — Random Forest importance ───────────────────────────────────
rf_embed = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
rf_embed.fit(X_train, y_train)
importances   = rf_embed.feature_importances_
emb_indices   = np.argsort(importances)[::-1][:k]
emb_feature_names = feature_names[emb_indices].tolist()

# ── 4) All features (no selection — upper-bound baseline) ────────────────────
all_indices = np.arange(n_features)

# Summary
method_map = {
    "GA (Genetic Algorithm)"         : ga_indices,
    "RFE (Recursive Feature Elim.)" : rfe_indices,
    "SelectKBest (Filter)"           : kbest_indices,
    "Embedded (RF Importance)"       : emb_indices,
    "All Features (No Selection)"    : all_indices,
}

print("\nFeature set summary:")
for name, idx in method_map.items():
    print(f"  {name:<40s}: {len(idx)} features")

## Cell 8 — Evaluate SVM and Decision Tree on Each Feature Set

In [ ]:
def _eval_classifier(name: str, clf, X_tr, X_te, y_tr, y_te) -> Dict:
    """Fit and score a classifier; return a metrics dict."""
    clf = deepcopy(clf)
    cv  = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring="accuracy")

    clf.fit(X_tr, y_tr)
    y_pred  = clf.predict(X_te)
    test_acc = float(accuracy_score(y_te, y_pred))
    prec     = float(precision_score(y_te, y_pred, zero_division=0))
    rec      = float(recall_score(y_te, y_pred, zero_division=0))
    f1       = float(f1_score(y_te, y_pred, zero_division=0))

    roc_auc = np.nan
    if hasattr(clf, "predict_proba"):
        y_score = clf.predict_proba(X_te)[:, 1]
        fpr, tpr, _ = roc_curve(y_te, y_score)
        roc_auc = float(auc(fpr, tpr))
    elif hasattr(clf, "decision_function"):
        raw = clf.decision_function(X_te)
        y_score = (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)
        fpr, tpr, _ = roc_curve(y_te, y_score)
        roc_auc = float(auc(fpr, tpr))

    return {
        "Classifier"   : name,
        "CV_Accuracy"  : float(np.mean(cv_scores)),
        "Test_Accuracy": test_acc,
        "Precision"    : prec,
        "Recall"       : rec,
        "F1_Score"     : f1,
        "ROC_AUC"      : roc_auc,
    }


# Classifiers to benchmark
classifiers = {
    "SVM (RBF)": SVC(
        C=2.0, kernel="rbf", gamma="scale",
        probability=True, random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        criterion="entropy", max_depth=10, random_state=RANDOM_STATE
    ),
}

results = []
print("\n=== [Phase 2] CLASSIFIER EVALUATION PER FEATURE SET ===")

for method_name, feat_idx in method_map.items():
    Xtr_sel = X_train[:, feat_idx]
    Xte_sel = X_test[:, feat_idx]

    for clf_name, clf in classifiers.items():
        print(f"  [{method_name}] + {clf_name} ...", end=" ", flush=True)
        row = _eval_classifier(clf_name, clf, Xtr_sel, Xte_sel, y_train, y_test)
        row["Feature_Method"] = method_name
        row["N_Features"]     = len(feat_idx)
        results.append(row)
        print(f"Acc={row['Test_Accuracy']:.4f}  F1={row['F1_Score']:.4f}")

results_df = pd.DataFrame(results)
print("\n=== FULL RESULTS TABLE ===")
print(results_df[["Feature_Method", "Classifier", "N_Features",
                  "CV_Accuracy", "Test_Accuracy", "F1_Score", "ROC_AUC"]].to_string(index=False))

## Cell 9 — Visualisations

### 9a — GA Evolution Curve

In [ ]:
SHOW_PLOTS = True   # set False to suppress all charts

history_df = pd.DataFrame(ga_selector.history_)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history_df["generation"], history_df["best_fitness"],
             marker="o", color="#2a9d8f", label="Best fitness")
axes[0].plot(history_df["generation"], history_df["avg_fitness"],
             marker="s", color="#457b9d", linestyle="--", label="Avg fitness")
axes[0].set_title("GA Fitness over Generations")
axes[0].set_xlabel("Generation")
axes[0].set_ylabel("Fitness (CV Acc − penalty)")
axes[0].legend()

axes[1].plot(history_df["generation"], history_df["n_selected"],
             marker="^", color="#e76f51")
axes[1].axhline(n_features, color="gray", linestyle=":", label="All features")
axes[1].set_title("Selected Feature Count over Generations")
axes[1].set_xlabel("Generation")
axes[1].set_ylabel("# Features Selected")
axes[1].legend()

plt.suptitle("Genetic Algorithm — Evolution History", fontsize=13)
plt.tight_layout()
_show_or_close(SHOW_PLOTS)

### 9b — Test Accuracy Comparison: GA vs. Baselines

In [ ]:
pivot_acc = results_df.pivot(index="Feature_Method", columns="Classifier", values="Test_Accuracy")
pivot_f1  = results_df.pivot(index="Feature_Method", columns="Classifier", values="F1_Score")

# Consistent ordering for the plot
method_order = list(method_map.keys())
pivot_acc = pivot_acc.reindex(method_order)
pivot_f1  = pivot_f1.reindex(method_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

x      = np.arange(len(method_order))
width  = 0.35
colors = ["#2a9d8f", "#e9c46a"]

for i, clf_col in enumerate(pivot_acc.columns):
    axes[0].bar(x + i * width - width / 2, pivot_acc[clf_col], width,
                label=clf_col, color=colors[i], alpha=0.88)
    axes[1].bar(x + i * width - width / 2, pivot_f1[clf_col], width,
                label=clf_col, color=colors[i], alpha=0.88)

for ax, title, ylabel in zip(
    axes,
    ["Test Accuracy by Feature Selection Method", "F1 Score by Feature Selection Method"],
    ["Test Accuracy", "F1 Score"],
):
    ax.set_xticks(x)
    ax.set_xticklabels(
        [m.replace(" (", "\n(") for m in method_order],
        rotation=20, ha="right", fontsize=9
    )
    ax.set_ylim(0.80, 1.01)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.axhline(y=pivot_acc.values.max() if title.startswith("Test") else pivot_f1.values.max(),
               color="red", linestyle=":", alpha=0.4, label="Best")

plt.tight_layout()
_show_or_close(SHOW_PLOTS)

### 9c — Feature Overlap Heatmap

How much do the feature subsets overlap? (Jaccard similarity)

In [ ]:
method_names_short = [
    "GA", "RFE", "SelectKBest", "Embedded RF", "All Features"
]
feat_sets  = [set(idx.tolist()) for idx in method_map.values()]
n_methods  = len(feat_sets)
jaccard    = np.zeros((n_methods, n_methods))

for i in range(n_methods):
    for j in range(n_methods):
        inter = len(feat_sets[i] & feat_sets[j])
        union = len(feat_sets[i] | feat_sets[j])
        jaccard[i, j] = inter / union if union > 0 else 1.0

plt.figure(figsize=(8, 6))
sns.heatmap(
    jaccard,
    annot=True, fmt=".2f",
    xticklabels=method_names_short,
    yticklabels=method_names_short,
    cmap="YlGnBu", vmin=0, vmax=1,
)
plt.title("Feature Set Overlap (Jaccard Similarity)")
plt.tight_layout()
_show_or_close(SHOW_PLOTS)

### 9d — ROC Curves (SVM, all feature methods)

In [ ]:
plt.figure(figsize=(9, 7))

svm_proto = SVC(C=2.0, kernel="rbf", gamma="scale",
                probability=True, random_state=RANDOM_STATE)

line_colors = ["#2a9d8f", "#e76f51", "#457b9d", "#e9c46a", "#264653"]

for (method_name, feat_idx), color in zip(method_map.items(), line_colors):
    clf = deepcopy(svm_proto)
    clf.fit(X_train[:, feat_idx], y_train)
    y_score = clf.predict_proba(X_test[:, feat_idx])[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_auc = auc(fpr, tpr)
    label = f"{method_name.split('(')[0].strip()} ({len(feat_idx)} feat, AUC={roc_auc:.3f})"
    plt.plot(fpr, tpr, label=label, color=color)

plt.plot([0, 1], [0, 1], "k--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — SVM with Different Feature Selection Methods")
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
_show_or_close(SHOW_PLOTS)

### 9e — Feature Importance of GA-Selected Features (visualised via RF)

In [ ]:
rf_ga = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
rf_ga.fit(X_train[:, ga_indices], y_train)

imp = rf_ga.feature_importances_
sort_idx = np.argsort(imp)[::-1][:30]   # top-30 for readability

plt.figure(figsize=(12, 6))
plt.bar(range(len(sort_idx)), imp[sort_idx], color="#2a9d8f", alpha=0.88)
plt.xticks(
    range(len(sort_idx)),
    feature_names[ga_indices[sort_idx]],
    rotation=60, ha="right", fontsize=8,
)
plt.title("RF Feature Importances within GA-Selected Subset (top 30)")
plt.ylabel("Importance")
plt.tight_layout()
_show_or_close(SHOW_PLOTS)

## Cell 10 — Summary Table and Interpretation

In [ ]:
print("\n=== [Phase 2] FINAL SUMMARY ===")

# Best config per classifier
for clf_name in classifiers.keys():
    subset = results_df[results_df["Classifier"] == clf_name].copy()
    best   = subset.sort_values("Test_Accuracy", ascending=False).iloc[0]
    print(f"\n[{clf_name}] Best feature method: {best['Feature_Method']}")
    print(f"  Test Accuracy : {best['Test_Accuracy']:.4f}")
    print(f"  F1 Score      : {best['F1_Score']:.4f}")
    print(f"  ROC AUC       : {best['ROC_AUC']:.4f}")
    print(f"  # Features    : {best['N_Features']}")

print("\n")
print("─" * 70)
print("GA vs. RFE (direct comparison):")
for clf_name in classifiers.keys():
    ga_row  = results_df[(results_df["Classifier"] == clf_name) & 
                          (results_df["Feature_Method"].str.startswith("GA"))].iloc[0]
    rfe_row = results_df[(results_df["Classifier"] == clf_name) & 
                          (results_df["Feature_Method"].str.startswith("RFE"))].iloc[0]

    delta = ga_row["Test_Accuracy"] - rfe_row["Test_Accuracy"]
    winner = "GA" if delta >= 0 else "RFE"
    print(f"  {clf_name:<18s}: GA={ga_row['Test_Accuracy']:.4f}  "
          f"RFE={rfe_row['Test_Accuracy']:.4f}  Δ={delta:+.4f}  → {winner} wins")

print("─" * 70)
print("\nGA configuration used:")
for k, v in vars(ga_cfg).items():
    print(f"  {k:<22s}: {v}")

print("\n[Phase 2] Complete. All cells ran successfully.")

## Interpretation

### What the GA does differently from RFE

| Aspect | RFE | Genetic Algorithm |
|--------|-----|-------------------|
| Search strategy | Greedy — removes features one-by-one | Evolutionary — explores many subsets in parallel |
| Feature interaction | Does not model interactions | Population diversity can capture synergistic subsets |
| Fitness signal | Estimator coefficient weight | k-fold CV accuracy on the actual dataset |
| Computation cost | Low — O(n_features × fit) | High — O(pop × gen × CV × fit) |
| Global optimum | Prone to local optima | Better exploration of search space |

### Why GA-selected features may improve classifiers
- **Redundancy removal:** irrelevant features add noise for SVM kernels and inflate Decision Tree search space.
- **Dimensionality reduction:** fewer features → faster inference and less risk of overfitting.
- **Non-linear synergy:** GA evaluates subsets as a whole, capturing feature interactions that greedy methods miss.

### Limitations and future extensions
- GA runtime grows with population size and generations — use a smaller oracle (DT/LR) to keep it tractable.
- Multi-objective GA (NSGA-II) could jointly optimise accuracy *and* subset size on the Pareto front.
- Hybrid: initialise GA population from RFE/SelectKBest results for faster convergence.
